In [5]:
%load_ext autoreload
%autoreload 2

import os
import sys

if not os.getcwd().endswith("/quotaclimat"):
    os.chdir("../../../..")

repo_root_path = os.path.abspath(os.path.dirname(os.getcwd()))
if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)
repo_root_path



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


'/root/Workspace'

In [7]:
from quotaclimat.data_ingestion.advertising_detection.e01_download_audio import (
    download_audio,
)
from quotaclimat.data_ingestion.advertising_detection.e02_create_chunks import (
    Chunk,
    ChunkCreator,
)
from quotaclimat.data_ingestion.advertising_detection.e04_group_chunks import (
    ChunkGrouping,
    debug_pair,
)
from quotaclimat.data_ingestion.advertising_detection.tools.cache import LocalCache
from quotaclimat.data_ingestion.advertising_detection.e00_partition_window import (
    partition_week_program,
)
from quotaclimat.data_ingestion.advertising_detection.tools.visualizer.chunk_comparator import (
    generate_chunk_comparator,
)

import json
from datetime import timedelta

In [8]:
channel = "tf1"
start_date = "2025-05-05"

partition = partition_week_program(
    channel=channel,
    start_date=start_date,
    margin=timedelta(minutes=15),
)

with LocalCache(name="chunks", version="8f4b4c10c550365f") as chunk_cache:
    chunks: list[Chunk] = []
    for segment in partition:
        chunk_batch = json.loads(chunk_cache.get(segment.identifier + ".json"))
        chunks.extend([Chunk.from_dict(d) for d in chunk_batch])

def get_chunk_by_start(start_sec: float) -> Chunk | None:
    return next((c for c in chunks if abs(c.start_sec - start_sec) < 1), None)

def partition_of_chunk(chunk: Chunk):
    return next(
        (p for p in partition if p.start_date.timestamp() <= chunk.start_sec < p.end_date.timestamp()), None
    )

chunk_grouping = ChunkGrouping(
    similarity_threshold=0.05,
    min_matching_hashes=1,
    duration_tol=1.5,
    rms_tol=0.05,
    centroid_tol=0.05,
    zcr_tol=0.1,
)
chunk_creator = ChunkCreator(
    sr=22050,
    hop_length=512,
    n_mfcc=20,
    context_sec=1.0,
    novelty_smooth_sec=0.5,
    min_chunk_sec=1.0,
    sensitivity=0.25,
    silence_percentile=5.0,
    n_fft=2048,
    n_peaks=30,
    neighborhood=15,
    min_amplitude=0.01,
)

In [ ]:
chunk_a = get_chunk_by_start(1746420874.53)
chunk_b = get_chunk_by_start(1746680080.76)

debug_pair(
    chunk_a,
    chunk_b,
    chunk_grouping,
)

html = generate_chunk_comparator(
    chunk_a=chunk_a,
    chunk_b=chunk_b,
    audio_a_path=(await download_audio(partition_of_chunk(chunk_a)))[0],
    audio_b_path=(await download_audio(partition_of_chunk(chunk_b)))[0],
    segment_a_start_epoch=chunk_a.start_sec,
    segment_b_start_epoch=chunk_b.start_sec,
    chunk_creator=chunk_creator,  # optionnel, utilise les params par défaut sinon
    padding_sec=1.0,  # configurable
)

with open(".cache/comparison.html", "w") as f:
    f.write(html)


DEBUG: chunk pair grouping analysis
  A: [1746420874.53s – 1746420876.30s]  channel=tf1
  B: [1746680080.76s – 1746680082.75s]  channel=tf1

[1] Minimum duration filter (>= 0.5 s)
    A duration: 1.765s  ✓
    B duration: 1.997s  ✓

[2] Acoustic pre-filter (_features_compatible)
    duration |A-B| = 0.232s  (tol=1.5)  ✓
    energy_mean rel_diff = 0.0990  (tol=0.05)  ✗  (A=0.0249, B=0.0225)
    spectral_centroid rel_diff = 0.0097  (tol=0.05)  ✓  (A=2813.9, B=2786.7)
    zcr_mean rel_diff = 0.1721  (tol=0.1)  ✗  (A=0.1343, B=0.1112)
  → BLOCKED: acoustic pre-filter rejected this pair.
